<a href="https://colab.research.google.com/github/427paul/AI_Agent/blob/main/%5BBDA%5D_11%EC%A3%BC%EC%B0%A8_LLM_RAG_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12주차 실습: 임베딩 & 벡터 DB

```
오늘 목표: 텍스트를 숫자로 바꾸고, 의미적으로 비슷한 문장을 찾아보기

비유: 도서관에서 책을 찾는 방법
  키워드 검색: '강아지'라는 단어가 제목에 있는 책만 찾음
  의미  검색: '반려동물', '개', 'pet' 같은 책도 함께 찾음
              → 임베딩 + 벡터 DB가 하는 일입니다
```

| 실습 블록 | 내용 | 셀 |
| --- | --- | --- |
| 사전 준비 | 환경 설정 | `0️⃣` |
| 임베딩 이해 | 텍스트 → 숫자 벡터 변환 직접 확인 | `1️⃣~3️⃣` |
| 유사도 계산 | 코사인 유사도로 문장 비교 | `4️⃣~5️⃣` |
| 벡터 DB | Chroma로 검색 DB 구축 + 검색 | `6️⃣~8️⃣` |
| 키워드 vs 의미 검색 | 두 방식의 차이 체감 | `9️⃣~🔟` |
| 위니브마켓 적용 | 상품 검색 DB 구축 | `1️⃣1️⃣~1️⃣3️⃣` |
| 정리 | 셀프 체크 | `1️⃣4️⃣` |

> ⚠️ **시작 전 체크**: `0️⃣` 셀에서 API Key를 입력하세요.

---
## 0️⃣ 환경 설정

In [1]:
# langchain-chroma와 chromadb가 새로 추가됩니다
!pip install -q langchain langchain-google-genai langchain-chroma chromadb
print('✅ 설치 완료')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6

In [2]:
import os
os.environ['GOOGLE_API_KEY'] = ''  # <- 여기에 키 입력

from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 임베딩 모델 — 텍스트를 숫자 벡터로 변환해주는 모델
embedding_model = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

print('✅ 임베딩 모델 준비 완료')
print('   gemini-embedding-001 → 텍스트 1개를 3072개 숫자로 변환합니다')

✅ 임베딩 모델 준비 완료
   gemini-embedding-001 → 텍스트 1개를 3072개 숫자로 변환합니다


---
## 🔢 임베딩 이해 — 텍스트가 숫자로 바뀐다
### 1️⃣ 문장 하나를 벡터로 변환해보기

In [3]:
# 문장 하나를 벡터로 변환
sentence = '강아지가 공원에서 뛰어놀아요'

vector = embedding_model.embed_query(sentence)

print(f'원본 문장: "{sentence}"')
print(f'변환된 벡터 차원: {len(vector)}개 숫자')
print()
print('벡터의 앞 10개 숫자만 출력:')
print([round(v, 4) for v in vector[:10]])
print()
print('...(이하 3062개 더 있음)')
print()
print('이 3072개 숫자가 문장의 "의미"를 담고 있습니다.')
print('숫자 하나하나의 의미는 알 수 없지만, 비슷한 문장은 비슷한 숫자 패턴을 가집니다.')

원본 문장: "강아지가 공원에서 뛰어놀아요"
변환된 벡터 차원: 3072개 숫자

벡터의 앞 10개 숫자만 출력:
[-0.0047, 0.0201, 0.0265, -0.0725, -0.0036, 0.0204, -0.046, 0.001, 0.0183, -0.0152]

...(이하 3062개 더 있음)

이 3072개 숫자가 문장의 "의미"를 담고 있습니다.
숫자 하나하나의 의미는 알 수 없지만, 비슷한 문장은 비슷한 숫자 패턴을 가집니다.


### 2️⃣ 비슷한 문장 vs 다른 문장 — 벡터가 정말 다를까?

In [4]:
# 비슷한 문장과 전혀 다른 문장의 벡터를 비교해봅니다
sentences = [
    '강아지가 공원에서 뛰어놀아요',    # 기준 문장
    '개가 야외에서 달리고 있어요',      # 비슷한 문장 (동의어 사용)
    '고양이가 집에서 낮잠을 자요',      # 약간 다른 문장 (동물이지만 다른 상황)
    '오늘 주식 시장이 급락했습니다',    # 전혀 다른 문장
]

vectors = embedding_model.embed_documents(sentences)

print('각 문장의 벡터 앞 5개 숫자 비교:')
print()
for s, v in zip(sentences, vectors):
    print(f'"{s[:20]}..."')
    print(f'  벡터: [{', '.join([str(round(x,3)) for x in v[:5]])}, ...]')
    print()

print('비슷한 문장(1번, 2번)은 숫자 패턴이 유사하고,')
print('전혀 다른 문장(4번)은 숫자 패턴이 완전히 달라 보입니다.')

각 문장의 벡터 앞 5개 숫자 비교:

"강아지가 공원에서 뛰어놀아요..."
  벡터: [-0.014, 0.027, 0.027, -0.075, -0.005, ...]

"개가 야외에서 달리고 있어요..."
  벡터: [-0.013, 0.009, 0.027, -0.077, 0.001, ...]

"고양이가 집에서 낮잠을 자요..."
  벡터: [-0.026, 0.034, 0.007, -0.074, 0.006, ...]

"오늘 주식 시장이 급락했습니다..."
  벡터: [-0.013, 0.008, -0.009, -0.081, 0.005, ...]

비슷한 문장(1번, 2번)은 숫자 패턴이 유사하고,
전혀 다른 문장(4번)은 숫자 패턴이 완전히 달라 보입니다.


### 3️⃣ 임베딩 비유 — 지도 위의 위치로 시각화

In [5]:
# 768차원을 2차원으로 줄여서 시각적으로 확인해봅니다
# (실제 임베딩은 768차원이지만, 시각화를 위해 2차원으로 압축)

import numpy as np

test_sentences = [
    # 동물 그룹
    '강아지가 뛰어놀아요',
    '개가 달리고 있어요',
    '고양이가 낮잠을 자요',
    '고양이가 생선을 먹어요',
    # 음식 그룹
    '김치찌개가 맛있어요',
    '삼겹살 구워 먹을래요',
    '오늘 저녁은 파스타',
    # 기술 그룹
    '파이썬으로 코딩해요',
    '머신러닝 모델을 학습시켜요',
    '딥러닝 논문을 읽었어요',
]

print('주제별로 분류된 문장들의 임베딩 생성 중...')
vecs = np.array(embedding_model.embed_documents(test_sentences))

# PCA로 3072차원 → 2차원 압축 (시각화 목적)
mean = vecs.mean(axis=0)
centered = vecs - mean
cov = np.cov(centered.T)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
top2 = eigenvectors[:, -2:]
coords = centered @ top2

print()
print('2차원으로 압축한 좌표 (지도 위의 위치):')
print(f'{'문장':<22} {'X':>8} {'Y':>8}')
print('-' * 42)
groups = ['🐾동물']*4 + ['🍽️음식']*3 + ['💻기술']*3
for sentence, coord, group in zip(test_sentences, coords, groups):
    print(f'{group} {sentence[:12]:<14} {coord[0]:>8.3f} {coord[1]:>8.3f}')

print()
print('같은 그룹의 문장들은 좌표가 서로 가깝습니다.')
print('이것이 임베딩이 "의미를 좌표로 표현한다"는 뜻입니다.')

주제별로 분류된 문장들의 임베딩 생성 중...

2차원으로 압축한 좌표 (지도 위의 위치):
문장                            X        Y
------------------------------------------
🐾동물 강아지가 뛰어놀아요        0.173    0.332
🐾동물 개가 달리고 있어요        0.173    0.339
🐾동물 고양이가 낮잠을 자요      -0.166    0.167
🐾동물 고양이가 생선을 먹어요     -0.249    0.119
🍽️음식 김치찌개가 맛있어요       -0.151   -0.088
🍽️음식 삼겹살 구워 먹을래요      -0.179   -0.122
🍽️음식 오늘 저녁은 파스타       -0.158   -0.125
💻기술 파이썬으로 코딩해요        0.143   -0.228
💻기술 머신러닝 모델을 학습시      0.206   -0.191
💻기술 딥러닝 논문을 읽었어요      0.206   -0.202

같은 그룹의 문장들은 좌표가 서로 가깝습니다.
이것이 임베딩이 "의미를 좌표로 표현한다"는 뜻입니다.


---
## 📐 유사도 계산 — 두 문장이 얼마나 비슷한가?
### 4️⃣ 코사인 유사도 직접 계산하기

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """두 벡터의 코사인 유사도를 계산합니다 (0~1, 높을수록 유사)"""
    v1, v2 = np.array(vec1), np.array(vec2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

# 기준 문장
base = '강아지가 공원에서 뛰어놀아요'

# 비교 문장들
compare_sentences = [
    ('거의 같은 의미 (동의어)',    '개가 야외에서 달리고 있어요'),
    ('비슷한 주제 (동물)',         '고양이가 나무 위에서 놀아요'),
    ('약간 관련 (활동)',           '아이들이 공원에서 축구해요'),
    ('전혀 다른 주제',            '오늘 주식 시장이 급락했습니다'),
    ('완전히 다른 분야',           '양자역학의 슈뢰딩거 방정식'),
]

base_vec = embedding_model.embed_query(base)

print(f'기준 문장: "{base}"')
print()
print(f'{'비교 유형':<20} {'유사도':>6}  {'시각화'}')
print('-' * 55)

for label, sentence in compare_sentences:
    compare_vec = embedding_model.embed_query(sentence)
    similarity = cosine_similarity(base_vec, compare_vec)
    bar = '█' * int(similarity * 20)
    print(f'{label:<20} {similarity:>6.3f}  {bar}')
    print(f'  "{sentence[:25]}"')
    print()

print('유사도가 1에 가까울수록 의미가 비슷한 문장입니다.')

기준 문장: "강아지가 공원에서 뛰어놀아요"

비교 유형                   유사도  시각화
-------------------------------------------------------
거의 같은 의미 (동의어)        0.862  █████████████████
  "개가 야외에서 달리고 있어요"

비슷한 주제 (동물)           0.717  ██████████████
  "고양이가 나무 위에서 놀아요"

약간 관련 (활동)            0.804  ████████████████
  "아이들이 공원에서 축구해요"

전혀 다른 주제              0.579  ███████████
  "오늘 주식 시장이 급락했습니다"

완전히 다른 분야             0.534  ██████████
  "양자역학의 슈뢰딩거 방정식"

유사도가 1에 가까울수록 의미가 비슷한 문장입니다.


### 5️⃣ 흥미로운 패턴 — 의미 연산

In [ ]:
# 임베딩의 재미있는 특성: 의미 연산
# 유명한 예시: '왕' - '남자' + '여자' = '여왕'
# 우리도 비슷한 패턴을 확인해봅시다

pairs = [
    ('서울', '부산', '대한민국 도시들은 임베딩이 비슷할까?'),
    ('행복하다', '슬프다', '반대 감정은 유사도가 낮을까?'),
    ('사과', '배', '같은 과일은 비슷할까?'),
    ('사과', '자동차', '전혀 다른 사물은 유사도가 낮을까?'),
    ('I love you', '사랑해', '같은 의미의 영어/한국어는 비슷할까?'),
]

print('=== 임베딩 유사도 실험 ===')
print()
for word1, word2, question in pairs:
    v1 = embedding_model.embed_query(word1)
    v2 = embedding_model.embed_query(word2)
    sim = cosine_similarity(v1, v2)
    bar = '█' * int(sim * 15)
    print(f'{question}')
    print(f'  "{word1}" vs "{word2}" → 유사도: {sim:.3f}  {bar}')
    print()

=== 임베딩 유사도 실험 ===

대한민국 도시들은 임베딩이 비슷할까?
  "서울" vs "부산" → 유사도: 0.693  ██████████

반대 감정은 유사도가 낮을까?
  "행복하다" vs "슬프다" → 유사도: 0.736  ███████████

같은 과일은 비슷할까?
  "사과" vs "배" → 유사도: 0.666  █████████

전혀 다른 사물은 유사도가 낮을까?
  "사과" vs "자동차" → 유사도: 0.648  █████████

같은 의미의 영어/한국어는 비슷할까?
  "I love you" vs "사랑해" → 유사도: 0.740  ███████████



---
## 🗄️ 벡터 DB — Chroma로 검색 데이터베이스 만들기
### 6️⃣ Chroma에 문장 5개 저장하기

드디어 벡터 DB를 만들어봅니다. `from_texts()` 한 줄이
텍스트 → 벡터 변환 + DB 저장을 동시에 처리합니다.

In [ ]:
from langchain_chroma import Chroma

# 저장할 문서 5개 (위니브마켓 고객 문의 예시)
documents = [
    '주문한 상품이 배송이 너무 늦어요. 언제 도착하나요?',
    '제품이 불량인 것 같아요. 교환이나 환불 받고 싶어요.',
    '회원가입하고 싶은데 방법을 모르겠어요.',
    '포인트 적립은 어떻게 사용하나요?',
    '상품 사이즈가 맞지 않아서 반품하고 싶어요.',
]

print('벡터 DB 구축 중...')
print('(각 문장을 임베딩 모델로 변환 후 저장합니다)')
print()

# from_texts(): 텍스트 목록을 받아서
# ① 각 텍스트를 임베딩 모델로 벡터 변환
# ② 원문 텍스트 + 벡터를 Chroma에 저장
# 두 단계를 한 번에 처리합니다
vectorstore = Chroma.from_texts(
    texts=documents,
    embedding=embedding_model,
)

print(f'✅ 벡터 DB 구축 완료!')
print(f'   저장된 문서 수: {vectorstore._collection.count()}개')
print()
print('저장된 문서 목록:')
for i, doc in enumerate(documents):
    print(f'  [{i}] {doc}')

벡터 DB 구축 중...
(각 문장을 임베딩 모델로 변환 후 저장합니다)

✅ 벡터 DB 구축 완료!
   저장된 문서 수: 5개

저장된 문서 목록:
  [0] 주문한 상품이 배송이 너무 늦어요. 언제 도착하나요?
  [1] 제품이 불량인 것 같아요. 교환이나 환불 받고 싶어요.
  [2] 회원가입하고 싶은데 방법을 모르겠어요.
  [3] 포인트 적립은 어떻게 사용하나요?
  [4] 상품 사이즈가 맞지 않아서 반품하고 싶어요.


### 7️⃣ 유사도 검색 — 의미적으로 가장 가까운 문서 찾기

In [ ]:
# similarity_search: 쿼리와 가장 비슷한 문서를 반환
queries = [
    '택배가 아직 안 왔어요',          # '배송 늦음' 관련 쿼리
    '물건이 망가져서 왔어요',          # '불량/교환' 관련 쿼리
    '적립금 쓰는 방법 알려줘',         # '포인트' 관련 쿼리
    '옷이 너무 커서 돌려보내고 싶어요', # '반품' 관련 쿼리
]

print('=== 유사도 검색 결과 ===')
print()

for query in queries:
    print(f'쿼리: "{query}"')

    # k=2: 가장 유사한 문서를 2개 반환
    results = vectorstore.similarity_search(query, k=2)

    for rank, doc in enumerate(results, 1):
        print(f'  {rank}위: {doc.page_content}')
    print()

print('쿼리와 저장된 문서의 단어가 달라도 의미적으로 유사한 문서를 찾아냅니다!')

=== 유사도 검색 결과 ===

쿼리: "택배가 아직 안 왔어요"
  1위: 주문한 상품이 배송이 너무 늦어요. 언제 도착하나요?
  2위: 제품이 불량인 것 같아요. 교환이나 환불 받고 싶어요.

쿼리: "물건이 망가져서 왔어요"
  1위: 제품이 불량인 것 같아요. 교환이나 환불 받고 싶어요.
  2위: 상품 사이즈가 맞지 않아서 반품하고 싶어요.

쿼리: "적립금 쓰는 방법 알려줘"
  1위: 포인트 적립은 어떻게 사용하나요?
  2위: 회원가입하고 싶은데 방법을 모르겠어요.

쿼리: "옷이 너무 커서 돌려보내고 싶어요"
  1위: 상품 사이즈가 맞지 않아서 반품하고 싶어요.
  2위: 제품이 불량인 것 같아요. 교환이나 환불 받고 싶어요.

쿼리와 저장된 문서의 단어가 달라도 의미적으로 유사한 문서를 찾아냅니다!


### 8️⃣ 유사도 점수와 함께 검색하기

In [ ]:
# similarity_search_with_score: 유사도 점수도 함께 반환
# Chroma는 기본적으로 L2 거리를 반환합니다 (낮을수록 유사)

query = '배송이 언제 오나요?'

print(f'쿼리: "{query}"')
print()
print(f'{'순위':<4} {'L2 거리':>8}  {'문서'}')
print('-' * 65)

results_with_score = vectorstore.similarity_search_with_score(query, k=5)

for rank, (doc, score) in enumerate(results_with_score, 1):
    indicator = ' ← 가장 관련 높음' if rank == 1 else ''
    print(f'{rank:<4} {score:>8.4f}  {doc.page_content}{indicator}')

print()
print('L2 거리(점수)가 낮을수록 쿼리와 더 가깝습니다.')
print('1위 문서가 가장 유사하고, 5위로 갈수록 관련이 적어집니다.')

쿼리: "배송이 언제 오나요?"

순위      L2 거리  문서
-----------------------------------------------------------------
1      0.3723  주문한 상품이 배송이 너무 늦어요. 언제 도착하나요? ← 가장 관련 높음
2      0.7174  제품이 불량인 것 같아요. 교환이나 환불 받고 싶어요.
3      0.7344  포인트 적립은 어떻게 사용하나요?
4      0.7358  회원가입하고 싶은데 방법을 모르겠어요.
5      0.7434  상품 사이즈가 맞지 않아서 반품하고 싶어요.

L2 거리(점수)가 낮을수록 쿼리와 더 가깝습니다.
1위 문서가 가장 유사하고, 5위로 갈수록 관련이 적어집니다.


---
## 🔍 키워드 검색 vs 의미 검색
### 9️⃣ 두 방식의 차이를 직접 체감하기

In [ ]:
# 키워드 검색: 단어가 포함된 문서만 반환
def keyword_search(query, documents):
    """단어 매칭 방식 — 쿼리 단어가 포함된 문서를 반환"""
    query_words = set(query.replace(' ', ''))
    results = []
    for doc in documents:
        # 공통 글자 수로 간단히 점수 계산
        doc_words = set(doc.replace(' ', ''))
        overlap = len(query_words & doc_words)
        if overlap > 0:
            results.append((overlap, doc))
    results.sort(reverse=True)
    return [doc for _, doc in results[:3]]

# 테스트용 문서 모음
knowledge_base = [
    '배가 아프면 소화제를 드세요.',
    '복통이 심할 때는 병원에 가야 합니다.',
    '배(梨)는 가을에 맛있는 과일입니다.',
    '뱃멀미가 있을 때는 멀미약을 드세요.',
    '위장 장애 증상으로는 통증, 구역질이 있습니다.',
    '사과는 건강에 좋은 과일입니다.',
    '오늘 주식 시장이 좋았어요.',
]

query = '배가 아파요'

print(f'쿼리: "{query}"')
print()
print('[키워드 검색 결과] — "배", "아파" 글자가 포함된 문서만 찾음')
kw_results = keyword_search(query, knowledge_base)
for i, r in enumerate(kw_results, 1):
    print(f'  {i}. {r}')
print()

쿼리: "배가 아파요"

[키워드 검색 결과] — "배", "아파" 글자가 포함된 문서만 찾음
  1. 배가 아프면 소화제를 드세요.
  2. 뱃멀미가 있을 때는 멀미약을 드세요.
  3. 배(梨)는 가을에 맛있는 과일입니다.



In [ ]:
# 의미 검색
import time

knowledge_store = Chroma.from_texts(
    texts=knowledge_base,
    embedding=embedding_model,
)

print('[의미 검색 결과] — "배가 아프다"는 의미와 가까운 문서를 찾음')
sem_results = knowledge_store.similarity_search(query, k=3)
for i, doc in enumerate(sem_results, 1):
    print(f'  {i}. {doc.page_content}')
print()

print('비교 분석:')
print('키워드 검색: "배(梨)"가 들어간 문장도 히트됨 (과일 이야기인데)')
print('의미  검색: "복통", "위장 장애" 같은 동의어 문장도 찾아냄')

[의미 검색 결과] — "배가 아프다"는 의미와 가까운 문서를 찾음
  1. 배가 아프면 소화제를 드세요.
  2. 복통이 심할 때는 병원에 가야 합니다.
  3. 위장 장애 증상으로는 통증, 구역질이 있습니다.

비교 분석:
키워드 검색: "배(梨)"가 들어간 문장도 히트됨 (과일 이야기인데)
의미  검색: "복통", "위장 장애" 같은 동의어 문장도 찾아냄


### 🔟 메타데이터로 검색 결과 필터링

In [ ]:
# 메타데이터: 각 문서에 추가 정보(태그, 카테고리 등)를 붙일 수 있습니다

# 텍스트와 메타데이터를 함께 정의 (쌍으로 묶어두면 실수가 없음)
docs_and_meta = [
    ('배송이 늦어요. 언제 도착하나요?',    '배송'),
    ('택배 추적 방법 알려주세요.',          '배송'),
    ('불량품 교환 신청하고 싶어요.',        '교환/환불'),
    ('제품이 파손되어 왔어요.',             '교환/환불'),
    ('포인트 사용 방법이 뭔가요?',          '포인트'),
    ('적립금 유효기간이 얼마나 되나요?',    '포인트'),
]

store_with_meta = Chroma.from_texts(
    texts=[text for text, _ in docs_and_meta],
    embedding=embedding_model,
    metadatas=[{'category': cat} for _, cat in docs_and_meta],
)

query = '상품 받는 데 얼마나 걸려요?'

print(f'쿼리: "{query}"')
print()

# 카테고리 필터 없이 검색
print('[필터 없이] 전체 문서 중 검색:')
results_all = store_with_meta.similarity_search(query, k=2)
for doc in results_all:
    # .get()으로 안전하게 접근 — 메타데이터가 없는 문서도 처리 가능
    cat = doc.metadata.get('category', '미분류')
    print(f'  [{cat}] {doc.page_content}')
print()

# 배송 카테고리만 검색
print('[필터 적용] 배송 카테고리 문서만 검색:')
results_filtered = store_with_meta.similarity_search(
    query, k=2,
    filter={'category': '배송'}  # 메타데이터 필터
)
for doc in results_filtered:
    cat = doc.metadata.get('category', '미분류')
    print(f'  [{cat}] {doc.page_content}')
print()
print('같은 쿼리라도 필터를 걸면 해당 카테고리 문서만 반환됩니다.')


쿼리: "상품 받는 데 얼마나 걸려요?"

[필터 없이] 전체 문서 중 검색:
  [미분류] 주문한 상품이 배송이 너무 늦어요. 언제 도착하나요?
  [배송] 배송이 늦어요. 언제 도착하나요?

[필터 적용] 배송 카테고리 문서만 검색:
  [배송] 배송이 늦어요. 언제 도착하나요?
  [배송] 택배 추적 방법 알려주세요.

같은 쿼리라도 필터를 걸면 해당 카테고리 문서만 반환됩니다.


---
## 🛒 실전 적용 — 위니브마켓 상품 FAQ 검색 DB
### 1️⃣1️⃣ 실제 규모의 문서 DB 구축

In [ ]:
# 위니브마켓 고객 FAQ 문서 (실제 서비스처럼 구성)
faq_data = [
    # 배송 관련
    {'text': '일반 배송은 결제 완료 후 2~3 영업일 내 발송됩니다.',                    'category': '배송'},
    {'text': '제주도, 도서산간 지역은 추가 배송비 3,000원이 부과됩니다.',              'category': '배송'},
    {'text': '50,000원 이상 구매 시 무료 배송이 적용됩니다.',                          'category': '배송'},
    {'text': '배송 조회는 마이페이지 > 주문내역에서 확인할 수 있습니다.',               'category': '배송'},
    # 교환/환불 관련
    {'text': '상품 수령 후 7일 이내에 교환 및 환불 신청이 가능합니다.',                'category': '교환/환불'},
    {'text': '단순 변심에 의한 반품 시 왕복 배송비 6,000원이 고객 부담입니다.',        'category': '교환/환불'},
    {'text': '불량품, 오배송의 경우 배송비 없이 교환 또는 환불이 가능합니다.',          'category': '교환/환불'},
    {'text': '환불 금액은 카드 취소 기준 3~5 영업일 내에 처리됩니다.',                 'category': '교환/환불'},
    # 포인트/혜택 관련
    {'text': '구매 금액의 1%가 포인트로 적립되며 다음 구매에 사용 가능합니다.',        'category': '포인트'},
    {'text': '포인트 유효기간은 적립 후 1년입니다.',                                  'category': '포인트'},
    {'text': '회원가입 시 즉시 사용 가능한 3,000 포인트가 지급됩니다.',               'category': '포인트'},
    # 회원 관련
    {'text': '비밀번호 분실 시 이메일 인증을 통해 재설정할 수 있습니다.',              'category': '회원'},
    {'text': '회원 탈퇴 후 30일 이내에는 동일 이메일로 재가입이 불가합니다.',          'category': '회원'},
]

print(f'FAQ 문서 {len(faq_data)}개로 벡터 DB 구축 중...')

faq_store = Chroma.from_texts(
    texts=[item['text'] for item in faq_data],
    embedding=embedding_model,
    metadatas=[{'category': item['category']} for item in faq_data],
)

print(f'✅ FAQ 벡터 DB 구축 완료! (총 {faq_store._collection.count()}개 문서)')

FAQ 문서 13개로 벡터 DB 구축 중...
✅ FAQ 벡터 DB 구축 완료! (총 31개 문서)


### 1️⃣2️⃣ FAQ 검색 테스트

In [ ]:
# 다양한 표현으로 같은 내용을 찾을 수 있는지 테스트
test_queries = [
    ('배송은 얼마나 걸려요?',          None),           # 일반 검색
    ('택배가 아직 안 왔는데요',         None),           # 같은 의미, 다른 표현
    ('물건 돌려보내고 싶어요',          None),           # '반품' 의미
    ('교환 가능한가요?',               '교환/환불'),     # 카테고리 필터 적용
    ('포인트 얼마나 받아요?',           '포인트'),       # 카테고리 필터 적용
]

print('=== 위니브마켓 FAQ 검색 테스트 ===')
print()

for query, category_filter in test_queries:
    filter_info = f' [카테고리: {category_filter} 한정]' if category_filter else ''
    print(f'Q: "{query}"{filter_info}')

    search_kwargs = {'k': 2}
    if category_filter:
        search_kwargs['filter'] = {'category': category_filter}

    results = faq_store.similarity_search(query, **search_kwargs)
    for i, doc in enumerate(results, 1):
        print(f'  {i}위 [{doc.metadata.get("category", "미분류")}]: {doc.page_content}')
    print()

=== 위니브마켓 FAQ 검색 테스트 ===

Q: "배송은 얼마나 걸려요?"
  1위 [배송]: 배송이 늦어요. 언제 도착하나요?
  2위 [미분류]: 주문한 상품이 배송이 너무 늦어요. 언제 도착하나요?

Q: "택배가 아직 안 왔는데요"
  1위 [배송]: 배송이 늦어요. 언제 도착하나요?
  2위 [미분류]: 주문한 상품이 배송이 너무 늦어요. 언제 도착하나요?

Q: "물건 돌려보내고 싶어요"
  1위 [미분류]: 상품 사이즈가 맞지 않아서 반품하고 싶어요.
  2위 [교환/환불]: 불량품 교환 신청하고 싶어요.

Q: "교환 가능한가요?" [카테고리: 교환/환불 한정]
  1위 [교환/환불]: 상품 수령 후 7일 이내에 교환 및 환불 신청이 가능합니다.
  2위 [교환/환불]: 불량품, 오배송의 경우 배송비 없이 교환 또는 환불이 가능합니다.

Q: "포인트 얼마나 받아요?" [카테고리: 포인트 한정]
  1위 [포인트]: 포인트 사용 방법이 뭔가요?
  2위 [포인트]: 구매 금액의 1%가 포인트로 적립되며 다음 구매에 사용 가능합니다.



### 1️⃣3️⃣ 파일로 저장하고 다시 불러오기 (영구 저장)

In [ ]:
import os

# persist_directory를 지정하면 파일로 저장됩니다
PERSIST_DIR = './winiv_faq_db'

print(f'벡터 DB를 {PERSIST_DIR}에 저장하는 중...')

# 저장 (persist_directory 지정)
persistent_store = Chroma.from_texts(
    texts=[item['text'] for item in faq_data],
    embedding=embedding_model,
    metadatas=[{'category': item['category']} for item in faq_data],
    persist_directory=PERSIST_DIR,
)

print(f'✅ 저장 완료')
print(f'   저장 위치: {PERSIST_DIR}')
print(f'   파일 목록: {os.listdir(PERSIST_DIR)}')
print()

# 저장된 DB 다시 불러오기 (프로그램 재시작 후에도 사용 가능)
loaded_store = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embedding_model,
)

print(f'불러온 DB 문서 수: {loaded_store._collection.count()}개')
result = loaded_store.similarity_search('반품 정책이 어떻게 되나요?', k=1)
print(f'검색 테스트: {result[0].page_content}')
print()
print('프로그램을 껐다 켜도 persist_directory에서 다시 불러올 수 있습니다.')
print('13주차에서 이 DB에 LLM을 연결해 RAG를 완성합니다!')

벡터 DB를 ./winiv_faq_db에 저장하는 중...
✅ 저장 완료
   저장 위치: ./winiv_faq_db
   파일 목록: ['chroma.sqlite3', '1a686c6a-ab48-4235-a0dc-0439963648f9']

불러온 DB 문서 수: 13개
검색 테스트: 상품 수령 후 7일 이내에 교환 및 환불 신청이 가능합니다.

프로그램을 껐다 켜도 persist_directory에서 다시 불러올 수 있습니다.
13주차에서 이 DB에 LLM을 연결해 RAG를 완성합니다!


---
## 📊 오늘 배운 내용 정리

In [ ]:
checklist = [
    ('임베딩이 무엇인지 설명할 수 있다',
     '텍스트를 고차원 숫자 벡터로 변환하는 것. 의미가 비슷한 텍스트는 비슷한 벡터를 가짐'),
    ('코사인 유사도의 의미를 설명할 수 있다',
     '두 벡터의 방향이 얼마나 같은지. 1에 가까울수록 유사, 0에 가까울수록 무관'),
    ('키워드 검색과 의미 검색의 차이를 설명할 수 있다',
     '키워드: 단어 매칭(동의어 미검색), 의미: 벡터 유사도(동의어/문맥 이해)'),
    ('Chroma.from_texts()로 벡터 DB를 구축할 수 있다',
     'texts와 embedding을 넘기면 변환+저장 한 번에 처리됨'),
    ('similarity_search()로 유사 문서를 검색할 수 있다',
     '쿼리와 의미적으로 가장 가까운 k개 문서를 반환'),
]

print('11주차 핵심 개념 셀프 체크')
print()
for i, (q, a) in enumerate(checklist, 1):
    print(f'{i}. {q}')
    print(f'   -> {a}')
    print()

print('='*55)
print('다음 주 (13주차): 기본 RAG 파이프라인')
print('오늘 만든 FAQ 벡터 DB에 LLM을 연결하면?')
print('-> 사용자 질문에 "위니브마켓 정책을 바탕으로" 정확하게 답하는')
print('   챗봇이 완성됩니다!')

12주차 핵심 개념 셀프 체크

1. 임베딩이 무엇인지 설명할 수 있다
   -> 텍스트를 고차원 숫자 벡터로 변환하는 것. 의미가 비슷한 텍스트는 비슷한 벡터를 가짐

2. 코사인 유사도의 의미를 설명할 수 있다
   -> 두 벡터의 방향이 얼마나 같은지. 1에 가까울수록 유사, 0에 가까울수록 무관

3. 키워드 검색과 의미 검색의 차이를 설명할 수 있다
   -> 키워드: 단어 매칭(동의어 미검색), 의미: 벡터 유사도(동의어/문맥 이해)

4. Chroma.from_texts()로 벡터 DB를 구축할 수 있다
   -> texts와 embedding을 넘기면 변환+저장 한 번에 처리됨

5. similarity_search()로 유사 문서를 검색할 수 있다
   -> 쿼리와 의미적으로 가장 가까운 k개 문서를 반환

다음 주 (13주차): 기본 RAG 파이프라인
오늘 만든 FAQ 벡터 DB에 LLM을 연결하면?
-> 사용자 질문에 "위니브마켓 정책을 바탕으로" 정확하게 답하는
   챗봇이 완성됩니다!
